In [1]:
pip install torch

In [2]:
pip install numpy

In [3]:
pip install scikit-learn

## Tensors

In [4]:
import torch

In [5]:
torch.cuda.is_available()

False

In [6]:
import numpy as np


In [7]:
arr=np.array([[1,2],[3,4]])
print(arr)

[[1 2]
 [3 4]]


In [8]:
tensor=torch.Tensor([[1,2],[3,4]])
print(tensor)

tensor([[1., 2.],
        [3., 4.]])


In [9]:
arr*5

array([[ 5, 10],
       [15, 20]])

In [10]:
tensor*5

tensor([[ 5., 10.],
        [15., 20.]])

In [11]:
np.ones((2,4))

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [13]:
torch.ones((2,4))

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [15]:
np.random.random((2,4))

array([[0.62783177, 0.04476812, 0.70370894, 0.60934762],
       [0.18110201, 0.97071327, 0.79731276, 0.56231041]])

In [16]:
torch.rand((2,4))

tensor([[0.8112, 0.9228, 0.5664, 0.3224],
        [0.0148, 0.3427, 0.0475, 0.2871]])

In [17]:
arr.shape

(2, 2)

In [18]:
arr.dtype

dtype('int64')

In [19]:
arr.device

'cpu'

In [21]:
tensor.shape

torch.Size([2, 2])

In [22]:
tensor.dtype

torch.float32

In [23]:
tensor.device

device(type='cpu')

In [24]:
a=torch.tensor([2.,3.],requires_grad=True)
b=torch.tensor([6.,4.],requires_grad=True)
f=3*a**3-b**2

In [25]:
f

tensor([-12.,  65.], grad_fn=<SubBackward0>)

In [26]:
f.backward(gradient=torch.tensor([1,1]))

In [28]:
print(a.grad)

tensor([36., 81.])


## In this notebook, I implemented a basic PyTorch pipeline where I created tensors, built a model, defined a loss function and optimizer, and trained the model using a training loop involving forward pass, loss computation, backpropagation, and weight updates.

In [55]:
import torch
import torch.nn as nn# Import neural network module
import torch.nn.functional as F
import torch.optim as optim# Import optimizer module
# DataLoader → used for batching
# TensorDataset → combines X and y into dataset
from  torch.utils.data import DataLoader,TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
# Standardize data (very important for ML models)
from sklearn.preprocessing import StandardScaler

In [56]:
X,y=load_breast_cancer(return_X_y=True)# Load dataset (features X and labels y)

In [57]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)# Split data into training and testing (80% train, 20% test)

In [58]:
scaler=StandardScaler()# Create scaler object

In [70]:
X_train_scaled=scaler.fit_transform(X_train)# Fit on training data and transform it
X_test_scaled=scaler.transform(X_test)# Only transform test data (do NOT fit again)
#We fit the scaler only on training data to avoid data leakage. The scaler learns mean and standard deviation from
#training data, and we apply the same transformation to test data so that the model sees consistent feature scaling without accessing test information.

In [37]:
X_test_scaled

array([[-0.00490448,  0.85835093,  0.06469797, ...,  0.88999676,
         3.63209004,  3.3933557 ],
       [-0.49110252, -1.54452154, -0.55822193, ..., -1.34480309,
        -1.00506921, -0.79609866],
       [-0.3723155 , -0.87433262, -0.34777602, ..., -0.23186088,
        -1.45657363, -0.74029189],
       ...,
       [-0.29496581, -0.18312732, -0.27161464, ...,  1.35954248,
         1.59791   ,  1.88826378],
       [-0.55463976,  0.02703645, -0.56984656, ..., -0.13824891,
        -0.38163962, -0.15009292],
       [ 1.64153888,  1.05683894,  2.04208782, ...,  1.98362222,
         4.07234702,  0.85893875]])

In [38]:
y_train

array([1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1,
       1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1,
       1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1,
       1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0,
       0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0,

In [71]:
# Convert numpy arrays → PyTorch tensors (float type required)
X_train_scaled_tensor=torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor=torch.from_numpy(X_test_scaled).float()
# Labels → add extra dimension using unsqueeze(1)
# because model output is (batch_size, 1)
#unsqueeze(1) → converts shape: (100,) → (100, 1)
y_train_tensor=torch.from_numpy(y_train).unsqueeze(1).float()
y_test_tensor=torch.from_numpy(y_test).unsqueeze(1).float()
'''
We convert NumPy arrays to PyTorch tensors because models operate on tensors. We use float() to match
the model’s expected datatype, and unsqueeze(1) is used to reshape labels from (N,) to (N,1) so they match the model output shape for loss computation.
'''

'\nWe convert NumPy arrays to PyTorch tensors because models operate on tensors. We use float() to match \nthe model’s expected datatype, and unsqueeze(1) is used to reshape labels from (N,) to (N,1) so they match the model output shape for loss computation.\n'

In [62]:
train_dataset=TensorDataset(X_train_scaled_tensor,y_train_tensor)# Combine X and y into dataset

In [63]:
X_train_scaled_tensor.shape# Check shape of input data

torch.Size([455, 30])

In [64]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)# Create DataLoader for batching

In [72]:
'''
I created a feedforward neural network using nn.Module. It has two hidden layers with ReLU activation to capture non-linear patterns, and a final sigmoid
layer to output probability for binary classification. The architecture maps 30 input features to a single output representing class probability
'''
class BCNet(nn.Module):
  def __init__(self):
    super(BCNet,self).__init__()
    # First layer: input (30 features) → 64 neurons
    self.fc1=nn.Linear(30,64)
    # Second layer: 64 → 32 neurons
    self.fc2=nn.Linear(64,32)
    # Output layer: 32 → 1 (binary classification)
    self.fc3=nn.Linear(32,1)

  def forward(self,x):
    # Apply ReLU activation after each hidden layer
    x=F.relu(self.fc1(x))
    x=F.relu(self.fc2(x))
    # Sigmoid for binary classification (output between 0 and 1)
    x=torch.sigmoid(self.fc3(x))
    return x


In [66]:
model=BCNet()# Create model instance

In [67]:
criterion=nn.BCELoss()# Binary Cross Entropy Loss (for binary classification)
optimizer=optim.Adam(model.parameters(),lr=0.001)# Adam optimizer (better than SGD in most cases)

In [68]:
epochs=20
for epoch in range(epochs):
  model.train()# Set model to training mode
  running_loss=0.0
  # Loop through batches
  for x_batch,y_batch in train_loader:
    optimizer.zero_grad()# Clear old gradients
    preds=model(x_batch)# Forward pass → predictions
    loss=criterion(preds,y_batch)# Calculate loss
    loss.backward()# Backpropagation → compute gradients
    optimizer.step()# Update weights
    running_loss+=loss.item()# Accumulate loss
  epoch_loss=running_loss/len(train_loader)# Average loss for epoch
  print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/20, Loss: 0.6581
Epoch 2/20, Loss: 0.5073
Epoch 3/20, Loss: 0.3318
Epoch 4/20, Loss: 0.2048
Epoch 5/20, Loss: 0.1306
Epoch 6/20, Loss: 0.1174
Epoch 7/20, Loss: 0.0835
Epoch 8/20, Loss: 0.0747
Epoch 9/20, Loss: 0.0681
Epoch 10/20, Loss: 0.0639
Epoch 11/20, Loss: 0.0633
Epoch 12/20, Loss: 0.0559
Epoch 13/20, Loss: 0.0607
Epoch 14/20, Loss: 0.0534
Epoch 15/20, Loss: 0.0483
Epoch 16/20, Loss: 0.0463
Epoch 17/20, Loss: 0.0440
Epoch 18/20, Loss: 0.0427
Epoch 19/20, Loss: 0.0415
Epoch 20/20, Loss: 0.0390


In [69]:
# Disable gradient computation (faster + no memory usage)
with torch.no_grad():
  model.eval()# Set model to evaluation mode
  preds=model(X_test_scaled_tensor)# Get predictions on test data
  loss=criterion(preds,y_test_tensor).item()# Calculate test loss
  print(f"Test Loss: {loss:.4f}")
  accuracy=((preds >= 0.5) ==y_test_tensor).float().mean().item()# Convert probabilities → 0 or 1
#

Test Loss: 0.0296


In [54]:
accuracy

0.9561403393745422